# 05. 글자 읽기(OCR) · QR 코드

> OCR 은 모델 **두 개**가 이어서 돈다: ① 글자가 **어디** 있는지 찾고 ② 찾은 자리의 글자를 **읽는다**.
> `easyocr` 이 둘을 묶어 준다. QR 은 AI 가 아니라 정해진 규칙(영상 처리)으로 읽는다.

### 이 시간에 배우는 것
1. easyocr 결과의 모양: (꼭짓점 4개, 글자, 확신도)
2. 확신도로 거르기, **읽는 순서**(위→아래, 왼→오른) 정하기
3. QR 만들기 · 읽기, 그리고 AI 와 규칙 기반의 차이

| | 글자 찾기 | 글자 읽기 |
|---|---|---|
| easyocr 안의 모델 | CRAFT (글자 영역 지도) | CRNN (한 줄을 왼→오른으로 읽기) |
| 결과 | 기울어진 사각형들 | 그 안의 글자 + 확신도 |

easyocr 은 PyTorch 로 **CPU** 에서 돈다 (OpenVINO 가 아니다).

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

## 2. 사진 — 글자가 보이게 찍는다 (책 표지, 안내문 등)

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 글자 읽기 — easyocr
- `Reader(["ko", "en"])` — 한글 + 영어. 언어를 늘리면 느려진다.
- `download_enabled=False` — 모델을 인터넷으로 받으러 가지 않는다. 모델은 `models/code/easyocr` 에 이미 있다.
- **흑백으로** 넣는다: 컬러 사진 경로가 일부 윈도우에서 오류를 내서 edge-lab 도 흑백으로 넣는다. 글자 읽기에는 색이 필요 없다.

In [ ]:
import easyocr

OCR_DIR = MODELS / "code" / "easyocr"
t0 = time.perf_counter()
reader = easyocr.Reader(["ko", "en"], gpu=False,
                        model_storage_directory=str(OCR_DIR), user_network_directory=str(OCR_DIR),
                        download_enabled=False)              # 인터넷으로 받으러 가지 않는다
print(f"올림 {time.perf_counter() - t0:.1f}초")

grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
t0 = time.perf_counter()
results = reader.readtext(grey)                             # [(꼭짓점 4개, 글자, 확신도), ...]
print(f"{len(results)}줄 · {(time.perf_counter() - t0) * 1000:.0f} ms")
for quad, text, conf in results:
    print(f"{conf:.2f}  {text}")

In [ ]:
view = frame.copy()
for quad, text, conf in results:
    pts = np.array(quad, dtype=np.int32)
    color = (0, 160, 0) if conf >= 0.5 else (0, 0, 255)      # 확신도 낮은 것은 빨강
    cv2.polylines(view, [pts], True, color, 2)               # 글자 자리는 기울어진 사각형(꼭짓점 4개)
show(view)

## 4. 거르고 · 순서대로 이어 붙이기
easyocr 이 돌려주는 순서는 읽는 순서가 아닐 수 있다. 사람이 읽는 순서로 맞춘다:
1. 확신도가 낮은 것(예: 0.5 미만)은 버린다.
2. 각 사각형의 **가운데 y** 로 줄을 나눈다 — y 가 글자 높이의 절반 안쪽이면 같은 줄.
3. 같은 줄 안에서는 **x** 순서로.

In [ ]:
def read_order(results, min_conf=0.5):
    items = []
    for quad, text, conf in results:
        if conf < min_conf:
            continue
        q = np.array(quad, float)
        items.append({"x": q[:, 0].min(), "cy": q[:, 1].mean(), "h": q[:, 1].max() - q[:, 1].min(), "text": text})
    items.sort(key=lambda it: it["cy"])
    lines, cur = [], []
    for it in items:
        if cur and abs(it["cy"] - cur[-1]["cy"]) > max(it["h"], cur[-1]["h"]) / 2:   # 다음 줄로 넘어감
            lines.append(cur); cur = []
        cur.append(it)
    if cur:
        lines.append(cur)
    return "\n".join(" ".join(it["text"] for it in sorted(line, key=lambda it: it["x"])) for line in lines)

print(read_order(results))

## 5. QR — 만들고 읽기
QR 은 세 모서리의 큰 네모(위치 찾기 무늬)로 자리를 잡고, 흑백 칸을 정해진 규칙으로 읽는다. 일부가 가려져도 **오류 정정** 덕에 읽힌다.
AI 처럼 "배운" 것이 아니라 규칙대로 계산하므로 결과가 맞거나 틀리거나 둘 중 하나다 (확신도가 없다).

In [ ]:
enc = cv2.QRCodeEncoder.create()
qr = enc.encode("https://example.com/edge-lab")             # 흑백 작은 그림 (칸 하나 = 1 픽셀)
print("QR 칸 수:", qr.shape)
qr = cv2.resize(qr, (qr.shape[1] * 8, qr.shape[0] * 8), interpolation=cv2.INTER_NEAREST)   # 칸을 8배로 — 흐려지지 않게 NEAREST
qr = cv2.copyMakeBorder(qr, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255)               # 둘레에 흰 여백(조용한 구역)
qr_bgr = cv2.cvtColor(qr, cv2.COLOR_GRAY2BGR)
show(qr_bgr, 240)

def read_qr(bgr):
    ok, texts, pts, _ = cv2.QRCodeDetector().detectAndDecodeMulti(bgr)
    if not ok or pts is None:
        return []
    return [(t, p.astype(int)) for t, p in zip(texts, pts) if t]

print("만든 QR:", [t for t, _ in read_qr(qr_bgr)])
print("웹캠 사진:", [t for t, _ in read_qr(frame)] or "QR 없음")

In [ ]:
# 가려도 읽힐까? — 가운데를 조금씩 더 가려 본다
for frac in (0.0, 0.05, 0.1, 0.2):
    t = qr_bgr.copy()
    hh, ww = t.shape[:2]
    s = int(min(hh, ww) * frac)
    cv2.rectangle(t, (ww // 2 - s // 2, hh // 2 - s // 2), (ww // 2 + s // 2, hh // 2 + s // 2), (0, 0, 0), -1)
    print(f"가운데 {frac:.0%} 가림 → {[x for x, _ in read_qr(t)] or '못 읽음'}")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| `Missing ... model` · 다운로드 시도 | `models/code/easyocr` 가 없다 → `setup.bat` |
| `too many values to unpack` | 컬러 사진을 넣었다 → 흑백으로 |
| 글자를 엉뚱하게 읽음 | 작거나 흐리다 → 가까이, 밝게. 확신도로 거른다 |
| QR 을 못 읽음 | 너무 작거나 기울었거나 흰 여백이 없다 |

## 정리
- OCR = 찾기 모델 + 읽기 모델. 결과는 (사각형, 글자, 확신도) 목록이고, 읽는 순서는 직접 정한다.
- 확신도는 "버릴지" 를 정하는 데 쓴다.
- QR 은 규칙 기반 — 오류 정정 범위 안에서는 가려져도 읽힌다.

## 해 볼 것
1. `reader.readtext(grey, detail=0)` 과 `paragraph=True` 는 무엇을 돌려주나?
2. 사진을 흐리게(`cv2.GaussianBlur`) · 작게(`cv2.resize`) 만들면 어디서부터 못 읽나?
3. QR 에 한글을 넣어 만들고 읽어 보자.
4. 영수증 사진에서 "합계" 줄의 숫자만 뽑아 보자.
   <details><summary>힌트</summary>

   `read_order(...)` 의 줄 중 "합계" 가 든 줄을 찾고, `re.findall(r"[\d,]+", line)` 으로 숫자를 꺼낸다.
   </details>

---
# 🛠 미니 프로젝트 — 명함 스캐너

**만들 것**: 명함(또는 안내문) 사진에서 글자를 읽어 **전화번호 · 이메일 · 이름 후보**를 뽑고, 연락처를 담은 **QR** 을 만들어 저장한다.
QR 을 휴대폰 카메라로 찍으면 연락처로 저장할 수 있다.

**쓰는 것**: `reader` · `read_order` (이 노트북), 정규식(`re`), `cv2.QRCodeEncoder`

| 찾을 것 | 정규식 |
|---|---|
| 휴대폰 | `01[016789][-. ]?\d{3,4}[-. ]?\d{4}` |
| 이메일 | `[\w.+-]+@[\w-]+\.[\w.]+` |

In [ ]:
import re, json

card = grab()                                     # 명함을 웹캠에 보여 주고 실행
text = read_order(reader.readtext(cv2.cvtColor(card, cv2.COLOR_BGR2GRAY)), min_conf=0.3)
phones = re.findall(r"01[016789][-. ]?\d{3,4}[-. ]?\d{4}", text)
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
lines = [l.strip() for l in text.splitlines() if l.strip()]
name = next((l for l in lines if re.fullmatch(r"[가-힣]{2,4}", l.replace(" ", ""))), lines[0] if lines else "")   # 한글 2~4자 줄을 이름 후보로

contact = {"이름": name, "전화": phones[:1], "이메일": emails[:1], "전체 글": lines}
print(json.dumps(contact, ensure_ascii=False, indent=2))

mecard = f"MECARD:N:{name};" + (f"TEL:{phones[0]};" if phones else "") + (f"EMAIL:{emails[0]};" if emails else "") + ";"
q = cv2.QRCodeEncoder.create().encode(mecard)
q = cv2.copyMakeBorder(cv2.resize(q, (q.shape[1] * 8, q.shape[0] * 8), interpolation=cv2.INTER_NEAREST), 32, 32, 32, 32, cv2.BORDER_CONSTANT, value=255)
cv2.imwrite("contact_qr.png", q)
Path("contact.json").write_text(json.dumps(contact, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장: contact.json · contact_qr.png  | QR 확인:", [t for t, _ in read_qr(cv2.cvtColor(q, cv2.COLOR_GRAY2BGR))])

**더 해 보기**
- 회사 전화(02-…, 031-…) 도 찾기
- 여러 장의 명함을 차례로 읽어 하나의 CSV 연락처 목록으로
- 이름 후보를 더 똑똑하게: 가장 큰 글자(사각형 높이가 가장 큰 줄)를 이름으로